# 05 — Preventing prior collapse in solution-augmented BKT

## Purpose

Notebook 04 treats the student's initial incorrect solution as an ordinary
false observation for every KC in the dialogue-wide KC union. That fit sends
all 142 KC priors to the `0.001` lower bound.

This notebook tests several ways of preventing that collapse. Each approach has
its own section containing:

1. a plain-language description of what changes;
2. the reason for testing it;
3. the assumption or limitation it introduces;
4. its own fit or prediction run;
5. parameter diagnostics and three evaluation protocols;
6. an interpretation before moving to the next approach.

The purpose is to understand the mechanisms, not to select whichever method
looks best on the test set.


## 1. Shared experimental contract

Every approach uses the same reformatted MathDial population and the same
notebook 04 solution representation:

- `solution` is marked `correct=False`;
- its KCs are the ordered union of all KCs appearing later in the dialogue;
- the solution is included in augmented BKT state updating;
- the solution itself is never scored;
- paper filtering and the final-turn correctness override are unchanged;
- retained dialogues contain the solution plus at least two usable tagged real
  turns.

Three evaluation protocols are reported independently in every approach:

| Protocol | Scored real responses | Earlier information retained |
| --- | --- | --- |
| All real turns | R1 and later | S0 updates augmented models |
| Paper-matched R2+ | R2 and later | S0 and R1 update augmented models |
| First real turn R1 | R1 only | Shows the immediate effect of S0 |

The paper's accuracy, AUC, and binary F1 are retained. Log loss and Brier score
are added because a prior-control method can change probability calibration
without changing a hard prediction.

### How to judge an approach

Preventing a prior from equalling `0.001` is not sufficient. For each approach
we ask:

- Did it produce meaningful variation in priors, or merely move every prior to
  another boundary?
- Did the fitting pressure move into learning, guess, or slip?
- Did held-out real-turn prediction improve?
- Did the method introduce a new mismatch or an arbitrary hyperparameter?

The hard floor `0.05` and shrinkage strength `20` are declared sensitivity
settings. They were not chosen from test performance.


In [1]:
import os
import sys
import json
import re
import time
from ast import literal_eval
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "data" / "misconception").exists():
        os.chdir(_candidate)
        break
else:
    raise FileNotFoundError(f"Could not find data/misconception above {_here}")

sys.path.insert(0, str(Path.cwd() / "extension"))
from scripts import bkt, bkt_prior_controls

DATA = Path("data/misconception")
MODELS = Path("extension/models")
RESULTS = Path("extension/results")
MODELS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def parse_kcs(value):
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = literal_eval(text)
    except (SyntaxError, ValueError):
        return []
    return parsed if isinstance(parsed, list) else []

def parse_correct(value):
    text = str(value).strip().lower()
    if text == "true":
        return True
    if text == "false":
        return False
    return None

def read_reformatted(split):
    return pd.read_csv(
        DATA / f"mathdial_{split}.csv",
        keep_default_na=False,
        converters={"kcs": parse_kcs},
    )

train_raw = read_reformatted("train")
test_raw = read_reformatted("test")
print("working directory:", Path.cwd())
print("raw dialogues — train:", train_raw.dialogue_id.nunique(),
      "test:", test_raw.dialogue_id.nunique())


working directory: /Users/tandon.utsav2/Desktop/Experiment_1
raw dialogues — train: 2253 test: 595


## 2. Reconstruct and audit the common population

The population is reconstructed once and reused without alteration in every
section. A difference in results can therefore come from the model treatment,
not from retaining different dialogues or target turns.


In [2]:
def prepare_solution_long(raw, split):
    typical_mask = (
        pd.to_numeric(raw["self-typical-confusion"]) >= 1
    ) & (
        pd.to_numeric(raw["self-typical-interactions"]) >= 1
    )
    typical = raw.loc[typical_mask].copy()

    rows = []
    n_failed = n_short = n_kept = 0
    n_real_tagged = n_solution_pseudo = 0

    for dialogue_id, group in typical.groupby("dialogue_id", sort=False):
        group = group.reset_index(drop=True)
        assert group.iloc[0]["turn"] == "solution"
        solution_kcs = group.iloc[0]["kcs"]
        assert parse_correct(group.iloc[0]["correct"]) is False

        real = []
        for _, record in group.iloc[1:].iterrows():
            match = re.search(r"\d+", str(record["turn"]))
            if not match:
                continue
            kcs = record["kcs"]
            correct = parse_correct(record["correct"])
            if not kcs:
                correct = None
            real.append({
                "turn": int(match.group()),
                "correct": correct,
                "kcs": kcs,
            })

        if not any(item["correct"] is not None or item["kcs"] for item in real):
            n_failed += 1
            continue

        if real and real[-1]["kcs"] and real[-1]["correct"] is not None:
            outcome = str(group.iloc[-1]["self-correctness"])
            if outcome == "Yes":
                real[-1]["correct"] = True
            elif outcome == "No":
                real[-1]["correct"] = False
            elif outcome == "Yes, but I had to reveal the answer":
                real[-1]["correct"] = None

        tagged_real = [
            item for item in real
            if item["correct"] is not None and item["kcs"]
        ]
        if int(bool(solution_kcs)) + len(tagged_real) < 3:
            n_short += 1
            continue

        n_kept += 1
        n_real_tagged += len(tagged_real)
        n_solution_pseudo += len(solution_kcs)
        for kc in solution_kcs:
            rows.append({
                "dialogue_idx": int(dialogue_id), "turn": 0,
                "unit": "solution", "real_rank": 0,
                "correct": 0, "kc": str(kc),
            })
        for real_rank, item in enumerate(tagged_real, start=1):
            for kc in item["kcs"]:
                rows.append({
                    "dialogue_idx": int(dialogue_id),
                    "turn": item["turn"],
                    "unit": f"turn {item['turn']}",
                    "real_rank": real_rank,
                    "correct": int(item["correct"]),
                    "kc": str(kc),
                })

    long_df = pd.DataFrame(rows)
    audit = {
        "split": split,
        "raw_dialogues": raw.dialogue_id.nunique(),
        "failed_removed": n_failed,
        "fewer_than_3_tagged_removed": n_short,
        "dialogues_kept": n_kept,
        "real_tagged_turns": n_real_tagged,
        "solution_pseudo_observations": n_solution_pseudo,
        "pseudo_observations": len(long_df),
        "distinct_kcs": long_df["kc"].nunique(),
    }
    return long_df, audit

train_long, train_audit = prepare_solution_long(train_raw, "train")
test_long, test_audit = prepare_solution_long(test_raw, "test")
audit = pd.DataFrame([train_audit, test_audit]).set_index("split")
print(audit.to_string())


       raw_dialogues  failed_removed  fewer_than_3_tagged_removed  dialogues_kept  real_tagged_turns  solution_pseudo_observations  pseudo_observations  distinct_kcs
split                                                                                                                                                                
train           2253              18                          185            2050              10448                          9268                33221           142
test             595               7                           73             515               2500                          2304                 8092            96


In [3]:
expected = {
    "train": {"failed_removed": 18, "fewer_than_3_tagged_removed": 185,
              "dialogues_kept": 2050, "real_tagged_turns": 10448,
              "solution_pseudo_observations": 9268,
              "pseudo_observations": 33221},
    "test": {"failed_removed": 7, "fewer_than_3_tagged_removed": 73,
             "dialogues_kept": 515, "real_tagged_turns": 2500,
             "solution_pseudo_observations": 2304,
             "pseudo_observations": 8092},
}
for split, checks in expected.items():
    for field, value in checks.items():
        assert int(audit.loc[split, field]) == value, (
            split, field, audit.loc[split, field], value
        )

for frame in [train_long, test_long]:
    assert not frame[["correct", "kc"]].isna().any().any()
    per_dialogue = frame.drop_duplicates(["dialogue_idx", "real_rank"])
    assert per_dialogue.groupby("dialogue_idx").size().min() >= 3
    assert per_dialogue.groupby("dialogue_idx")["real_rank"].min().eq(0).all()

print("population and sequence assertions passed")
print(f"training pseudo-observation correctness rate: {train_long.correct.mean():.4f}")


population and sequence assertions passed
training pseudo-observation correctness rate: 0.3581


## 3. Load reference models and define shared diagnostics

Notebook 03 supplies the correctness-only model and the reference priors.
Notebook 04 supplies the unprotected solution-augmented model. Both are loaded
from their saved JSON files so their predictions can be checked exactly against
their saved turn-level outputs.

The helper below prints the same parameter and metric fields in every approach
section. In particular, it reports whether preventing prior collapse causes
learning, guess, or slip to reach a boundary instead.


In [4]:
def load_fitted(path):
    with path.open() as handle:
        payload = json.load(handle)
    return payload, bkt.FittedBKT(payload["per_skill"], payload["fallback"])

original_payload, original_model = load_fitted(MODELS / "bkt_original.json")
unprotected_payload, unprotected_model = load_fitted(
    MODELS / "bkt_solution_baseline.json"
)

training_skills = sorted(train_long["kc"].astype(str).unique())
original_prior_targets = bkt_prior_controls.reference_priors(
    training_skills,
    original_payload["per_skill"],
    original_payload["fallback"],
)
print("training KCs:", len(training_skills))
print("original fallback prior:", round(original_payload["fallback"]["prior"], 4))
print("unprotected fallback prior:", round(unprotected_payload["fallback"]["prior"], 4))


training KCs: 142
original fallback prior: 0.4882
unprotected fallback prior: 0.001


In [5]:
FIT_CONFIG = {"n_restarts": 5, "max_iter": 100, "tol": 1e-4, "seed": 221}
CONTROL_CONFIGS = {
    "hard_floor_005": {
        "policy": "hard_floor",
        "prior_floor": 0.05,
        "shrinkage_strength": 0.0,
    },
    "fixed_original_prior": {
        "policy": "fixed",
        "prior_floor": 0.001,
        "shrinkage_strength": 0.0,
    },
    "shrinkage_k20": {
        "policy": "shrinkage",
        "prior_floor": 0.001,
        "shrinkage_strength": 20.0,
    },
}

real_test_long = test_long[test_long["unit"] != "solution"].copy()
target_series = pd.Series(original_prior_targets).reindex(training_skills)

models = {}
turn_tables = {}
diagnostic_rows = {}
metric_tables = {}
fit_times = {}

DISPLAY_METRICS = [
    "n",
    "accuracy",
    "auc",
    "f1",
    "log_loss",
    "brier",
    "predicted_positive_rate",
    "mean_probability",
]


def parameter_frame(model):
    return pd.DataFrame(
        {skill: model.params_for(skill) for skill in training_skills}
    ).T[list(bkt.PARAM_NAMES)]


def parameter_diagnostic(name, model):
    params = parameter_frame(model)
    boundary = (
        np.isclose(params["prior"], 0.001)
        | np.isclose(params["prior"], 0.999)
        | np.isclose(params["learns"], 0.001)
        | np.isclose(params["learns"], 0.999)
        | np.isclose(params["guesses"], 0.001)
        | np.isclose(params["guesses"], 0.49)
        | np.isclose(params["slips"], 0.001)
        | np.isclose(params["slips"], 0.49)
    )
    return {
        "model": name,
        "prior_min": params["prior"].min(),
        "prior_median": params["prior"].median(),
        "prior_max": params["prior"].max(),
        "priors_at_0001": int(np.isclose(params["prior"], 0.001).sum()),
        "priors_at_005": int(np.isclose(params["prior"], 0.05).sum()),
        "median_abs_prior_shift_from_03": (
            params["prior"] - target_series
        ).abs().median(),
        "learning_median": params["learns"].median(),
        "guess_median": params["guesses"].median(),
        "slip_median": params["slips"].median(),
        "kcs_with_any_boundary_parameter": int(boundary.sum()),
    }


def aggregate_real_turns(pseudo_predictions, prediction_name):
    real = pseudo_predictions[pseudo_predictions["unit"] != "solution"].copy()
    assert real.groupby(["dialogue_idx", "turn"])["correct"].nunique().eq(1).all()
    return (
        real.groupby(
            ["dialogue_idx", "turn", "unit", "real_rank"], sort=False
        )
        .agg(
            **{prediction_name: ("pred", "mean")},
            correct=("correct", "first"),
        )
        .reset_index()
    )


def score_mask(frame, prediction_column, mask):
    scored = frame.loc[mask]
    labels = scored["correct"].to_numpy(dtype=int)
    probabilities = scored[prediction_column].to_numpy(dtype=float)
    clipped = np.clip(probabilities, 1e-12, 1 - 1e-12)
    hard = np.round(probabilities).astype(int)
    return {
        "n": len(scored),
        "dialogues": scored["dialogue_idx"].nunique(),
        "accuracy": accuracy_score(labels, hard),
        "auc": roc_auc_score(labels, probabilities),
        "f1": f1_score(labels, hard, pos_label=1, zero_division=0),
        "log_loss": log_loss(labels, clipped, labels=[0, 1]),
        "brier": brier_score_loss(labels, clipped),
        "observed_correct_rate": labels.mean(),
        "predicted_positive_rate": hard.mean(),
        "mean_probability": probabilities.mean(),
    }


def evaluate_one(name, model, prediction_data):
    pseudo = model.predict_long(prediction_data)
    turns = aggregate_real_turns(pseudo, name)
    assert len(turns) == 2500
    assert turns["dialogue_idx"].nunique() == 515
    masks = {
        "all_real_turns": turns["real_rank"] >= 1,
        "paper_matched_R2_plus": turns["real_rank"] >= 2,
        "first_real_turn_R1": turns["real_rank"] == 1,
    }
    rows = [
        {
            "protocol": protocol,
            "model": name,
            **score_mask(turns, name, mask),
        }
        for protocol, mask in masks.items()
    ]
    return turns, pd.DataFrame(rows)


def test_and_show(name, model, prediction_data):
    diagnostic = parameter_diagnostic(name, model)
    turns, approach_metrics = evaluate_one(name, model, prediction_data)
    print("Parameter diagnostics")
    print(pd.DataFrame([diagnostic]).set_index("model").round(4).to_string())
    print("\nEvaluation")
    print(
        approach_metrics.set_index("protocol")[DISPLAY_METRICS]
        .round(4)
        .to_string()
    )
    return turns, diagnostic, approach_metrics


## 4. Reference — original correctness-only BKT

### What the approach is

This is notebook 03's effective BKT baseline. It is fitted only on usable real
dialogue responses. S0 is neither a training observation nor an inference
update.

### Why it is tested

It is not a prior-collapse prevention method. It is the anchor that answers:

> Does any attempt to include the false solution improve on the model before
> that solution was introduced?

It also supplies the prior used by the fixed and shrinkage approaches. A
stabilized augmented model that still underperforms this reference has repaired
a parameter symptom, not the predictive problem.


In [6]:
name = "original_bkt"
models[name] = original_model
(
    turn_tables[name],
    diagnostic_rows[name],
    metric_tables[name],
) = test_and_show(name, models[name], real_test_long)


Parameter diagnostics
              prior_min  prior_median  prior_max  priors_at_0001  priors_at_005  median_abs_prior_shift_from_03  learning_median  guess_median  slip_median  kcs_with_any_boundary_parameter
model                                                                                                                                                                                       
original_bkt      0.001        0.4882      0.999              17              0                             0.0           0.0475        0.2938       0.2468                               81

Evaluation
                          n  accuracy     auc      f1  log_loss   brier  predicted_positive_rate  mean_probability
protocol                                                                                                          
all_real_turns         2500    0.5980  0.6278  0.5571    0.6648  0.2362                   0.4480            0.4921
paper_matched_R2_plus  1985    0.6050  0.6402  0.5556 

### Analysis of the original reference

The paper-matched result is accuracy `0.6050`, AUC `0.6402`, and log loss
`0.6604`. Its median prior is `0.4882` and median learning parameter is `0.0475`.

This model has boundary estimates of its own—81 of 142 reported KC parameter
vectors contain at least one boundary value—so it is not presented as perfectly
identified. It nevertheless provides the strongest overall held-out prediction
among the approaches in this notebook and defines the performance that a
solution-augmented alternative must justify.


## 5. Approach 1 — unprotected solution-augmented BKT

### What the approach is

This is notebook 04 without any new prior protection. Every dialogue–KC history
begins with a standard false observation from S0. The model jointly estimates
prior, learning, guess, and slip from those augmented sequences.

### Why it is tested

This is the failure reference. It establishes:

- the size of the prior collapse;
- how the solution changes the other parameters;
- the performance that each prevention method should improve upon.

No extra assumption is added beyond notebook 04, but its substantive assumption
is strong: every KC found anywhere later in the dialogue is treated as though it
was attempted incorrectly in S0.


In [7]:
name = "unprotected_solution_bkt"
models[name] = unprotected_model
(
    turn_tables[name],
    diagnostic_rows[name],
    metric_tables[name],
) = test_and_show(name, models[name], test_long)


Parameter diagnostics
                          prior_min  prior_median  prior_max  priors_at_0001  priors_at_005  median_abs_prior_shift_from_03  learning_median  guess_median  slip_median  kcs_with_any_boundary_parameter
model                                                                                                                                                                                                   
unprotected_solution_bkt      0.001         0.001      0.001             142              0                          0.4872           0.7305         0.001       0.4158                              142

Evaluation
                          n  accuracy     auc      f1  log_loss   brier  predicted_positive_rate  mean_probability
protocol                                                                                                          
all_real_turns         2500    0.5312  0.5843  0.5936    0.6899  0.2480                   0.6940            0.5099
paper_matched_R2_p

### Analysis of the unprotected approach

All 142 priors equal the `0.001` lower bound. Median learning rises to `0.7305`
and median guess falls to `0.001`. The model explains a common false beginning
followed by later mixed responses as almost no initial mastery, rapid learning,
and almost no chance of guessing correctly.

On paper-matched R2+, AUC falls from `0.6402` to `0.5640` and log loss worsens
from `0.6604` to `0.6954`. F1 rises to `0.6064`, but the model predicts correct
for 77.4% of targets. That threshold behavior does not offset the loss in
ranking, accuracy, and calibration.

This section confirms both the numerical symptom and the predictive degradation
that the remaining approaches attempt to address.


## 6. Approach 2 — apply the solution only during inference

### What the approach is

The model keeps every notebook 03 parameter unchanged. During test inference,
however, the false S0 pseudo-observations are placed before the real responses.
They update mastery and are followed by the normal learning transition.

### Why it is tested

This separates two possible causes of notebook 04's degradation:

1. retraining parameters on sequences that always begin false;
2. the false solution state update itself.

Because no augmented refit occurs, the priors cannot newly collapse. The cost is
a **train–inference mismatch**: the model is asked to process a kind of initial
observation that it never saw during fitting.

If this approach performed like the original, retraining would be the main
problem. If it still degraded, the forced false update itself would also be
harmful.


In [8]:
name = "inference_only_solution_update"
models[name] = original_model
(
    turn_tables[name],
    diagnostic_rows[name],
    metric_tables[name],
) = test_and_show(name, models[name], test_long)


Parameter diagnostics
                                prior_min  prior_median  prior_max  priors_at_0001  priors_at_005  median_abs_prior_shift_from_03  learning_median  guess_median  slip_median  kcs_with_any_boundary_parameter
model                                                                                                                                                                                                         
inference_only_solution_update      0.001        0.4882      0.999              17              0                             0.0           0.0475        0.2938       0.2468                               81

Evaluation
                          n  accuracy     auc      f1  log_loss   brier  predicted_positive_rate  mean_probability
protocol                                                                                                          
all_real_turns         2500    0.5800  0.6135  0.3446    0.6728  0.2399                   0.1812            0.4338


### Analysis of the inference-only approach

Its parameters are exactly the original model's parameters, so there is no new
prior collapse. It is the least damaging solution-augmented variant on
paper-matched AUC (`0.6212`) and log loss (`0.6746`), but it remains worse than
the original model.

At R1 it predicts correct for only 2.1% of turns and obtains F1 `0.0636`. The
false union update sharply reduces mastery before the first real response. Its
effect weakens after real observations accumulate, but paper-matched accuracy
still falls to `0.5748`.

This result shows that preventing refitted-prior collapse is not enough: the
solution-as-false-update representation is itself disruptive.


## 7. Approach 3 — impose a hard prior floor

### What the approach is

The prior M-step is clipped to:

\[
\pi_k \in [0.05, 0.999]
\]

instead of notebook 04's lower bound of `0.001`. Learning, guess, and slip are
still freely re-estimated within their usual bounds.

### Why it is tested

This is the simplest mechanical prevention method. It tests whether notebook
04 performs poorly mainly because the prior becomes *too close to zero*.

A useful floor would allow differentiated priors above `0.05` and improve
predictions. If every prior simply lands at `0.05`, the procedure has moved the
same collapse to a new arbitrary boundary. The value `0.05` is illustrative and
was not tuned on the test set.


In [9]:
name = "hard_floor_005"
config = CONTROL_CONFIGS[name]
started = time.time()
models[name] = bkt_prior_controls.fit_bkt_prior_control(
    train_long,
    reference_prior_by_skill=original_prior_targets,
    reference_fallback_prior=float(original_payload["fallback"]["prior"]),
    **FIT_CONFIG,
    **config,
)
fit_times[name] = time.time() - started
print(f"fit time: {fit_times[name]:.1f}s\n")
(
    turn_tables[name],
    diagnostic_rows[name],
    metric_tables[name],
) = test_and_show(name, models[name], test_long)


[prior_control] policy=hard_floor; fitted=131; degenerate=11
fit time: 94.4s

Parameter diagnostics
                prior_min  prior_median  prior_max  priors_at_0001  priors_at_005  median_abs_prior_shift_from_03  learning_median  guess_median  slip_median  kcs_with_any_boundary_parameter
model                                                                                                                                                                                         
hard_floor_005       0.05          0.05       0.05               0            142                          0.4382            0.811         0.001        0.445                              129

Evaluation
                          n  accuracy     auc      f1  log_loss   brier  predicted_positive_rate  mean_probability
protocol                                                                                                          
all_real_turns         2500    0.5336  0.5879  0.5932    0.6890  0.2476              

### Analysis of the hard-floor approach

All 142 priors equal exactly `0.05`. The approach therefore relocates the
collapse rather than producing differentiated KC priors. Median learning rises
further to `0.8110`, median guess remains `0.001`, and 129 KC parameter vectors
contain at least one boundary value.

Paper-matched AUC is `0.5684`, only `0.0044` above the unprotected model and
well below the original `0.6402`. Log loss remains poor at `0.6942`.

The experiment demonstrates why a higher clip is a safeguard rather than a
scientific solution: its chosen boundary becomes the new answer, while the
underlying conflict remains.


## 8. Approach 4 — fix priors to notebook 03

### What the approach is

For each KC:

\[
\pi_k^{05} = \pi_k^{03}.
\]

The original prior is held constant throughout EM. Learning, guess, and slip
are refitted on the full solution-augmented training sequences.

### Why it is tested

This is the strongest direct protection. It asks:

> If the ordinary BKT starting-mastery distribution is preserved exactly, can
> the remaining parameters absorb S0 without losing predictive performance?

It is also a diagnostic for parameter transfer. If priors remain sensible but
learning, guess, or slip become extreme, the forced solution evidence is still
in tension with the later response sequences.

The limitation is that the prior is no longer estimated from the augmented
likelihood. Its value is imposed from a separate correctness-only fit.


In [10]:
name = "fixed_original_prior"
config = CONTROL_CONFIGS[name]
started = time.time()
models[name] = bkt_prior_controls.fit_bkt_prior_control(
    train_long,
    reference_prior_by_skill=original_prior_targets,
    reference_fallback_prior=float(original_payload["fallback"]["prior"]),
    **FIT_CONFIG,
    **config,
)
fit_times[name] = time.time() - started
print(f"fit time: {fit_times[name]:.1f}s\n")
(
    turn_tables[name],
    diagnostic_rows[name],
    metric_tables[name],
) = test_and_show(name, models[name], test_long)

fixed_priors = parameter_frame(models[name])["prior"]
assert np.allclose(fixed_priors, target_series, atol=1e-12)
print("\nall fixed priors exactly match notebook 03")


[prior_control] policy=fixed; fitted=131; degenerate=11
fit time: 78.5s

Parameter diagnostics
                      prior_min  prior_median  prior_max  priors_at_0001  priors_at_005  median_abs_prior_shift_from_03  learning_median  guess_median  slip_median  kcs_with_any_boundary_parameter
model                                                                                                                                                                                               
fixed_original_prior      0.001        0.4882      0.999              17              0                             0.0           0.7598         0.001         0.49                              131

Evaluation
                          n  accuracy     auc      f1  log_loss   brier  predicted_positive_rate  mean_probability
protocol                                                                                                          
all_real_turns         2500    0.5548  0.5952  0.5664    0.6821  0.2446 

### Analysis of the fixed-prior approach

The prior distribution is preserved exactly, with median `0.4882` and zero
median absolute shift from notebook 03. The pressure moves elsewhere: median
learning becomes `0.7598`, median guess reaches `0.001`, median slip reaches its
`0.49` ceiling, and 131 KC parameter vectors contain a boundary value.

Paper-matched AUC improves over the unprotected model from `0.5640` to `0.5826`,
but remains far below the original `0.6402`. Log loss is `0.6860`.

Fixing the prior prevents the requested collapse, but the extreme remaining
parameters show that it cannot reconcile the forced false KC union with later
responses. The conflict has been transferred, not removed.


## 9. Approach 5 — shrink priors toward notebook 03

### What the approach is

Shrinkage combines the augmented-data posterior counts with a reference centred
on notebook 03:

\[
\pi_k^{new}
=
\frac{
\sum_d P(L_{d,k,0}=1\mid Y)
+ \kappa\pi_k^{03}
}{
N_k+\kappa
},
\qquad \kappa=20.
\]

The strength \(\kappa=20\) acts like 20 reference pseudo-dialogues for each KC.
At \(\kappa=0\), the method becomes the unprotected fit. As \(\kappa\) becomes
very large, it approaches the fixed-prior approach.

### Why it is tested

Shrinkage is a compromise:

- augmented data may move a prior when evidence is strong;
- rare KCs receive more relative protection;
- no KC prior must be fixed completely.

This is more statistically motivated than a universal hard floor, but the
strength is still a hyperparameter. Here it is a declared sensitivity setting,
not a test-selected optimum.


In [11]:
name = "shrinkage_k20"
config = CONTROL_CONFIGS[name]
started = time.time()
models[name] = bkt_prior_controls.fit_bkt_prior_control(
    train_long,
    reference_prior_by_skill=original_prior_targets,
    reference_fallback_prior=float(original_payload["fallback"]["prior"]),
    **FIT_CONFIG,
    **config,
)
fit_times[name] = time.time() - started
print(f"fit time: {fit_times[name]:.1f}s\n")
(
    turn_tables[name],
    diagnostic_rows[name],
    metric_tables[name],
) = test_and_show(name, models[name], test_long)


[prior_control] policy=shrinkage; fitted=131; degenerate=11
fit time: 93.2s

Parameter diagnostics
               prior_min  prior_median  prior_max  priors_at_0001  priors_at_005  median_abs_prior_shift_from_03  learning_median  guess_median  slip_median  kcs_with_any_boundary_parameter
model                                                                                                                                                                                        
shrinkage_k20      0.001        0.2441     0.9986              17              0                          0.0286           0.8939         0.001       0.4685                              131

Evaluation
                          n  accuracy     auc      f1  log_loss   brier  predicted_positive_rate  mean_probability
protocol                                                                                                          
all_real_turns         2500    0.5380  0.5973  0.5969    0.6836  0.2454                  

### Analysis of the shrinkage approach

The median prior is `0.2441`, rather than the unprotected `0.001`, and the
median absolute shift from notebook 03 is only `0.0286`. Unlike the hard floor,
the priors remain differentiated.

The remaining parameters are still extreme: median learning is `0.8939`,
median guess is `0.001`, median slip is `0.4685`, and 131 KC vectors contain a
boundary parameter. Paper-matched AUC is `0.5786` and log loss is `0.6875`.

At R1, shrinkage obtains AUC `0.6246`, slightly above the original model's
`0.6205`. That local benefit does not persist: all-real and R2+ performance
remain worse. Shrinkage successfully stabilizes the prior distribution, but
does not repair the overall solution-union observation model.


## 10. Side-by-side comparison and regression checks

The individual sections establish how each method behaves. This final table
places the same results side by side without selecting a winner from test
performance.

The main conclusions to check are:

1. the hard floor moves every prior to another boundary;
2. fixed and shrinkage priors transfer pressure into other parameters;
3. inference-only updating is least damaging by paper-matched AUC but introduces
   train–inference mismatch;
4. the original correctness-only model remains strongest overall.


In [12]:
MODEL_ORDER = [
    "original_bkt",
    "unprotected_solution_bkt",
    "inference_only_solution_update",
    "hard_floor_005",
    "fixed_original_prior",
    "shrinkage_k20",
]

parameter_diagnostics = pd.DataFrame(
    [diagnostic_rows[name] for name in MODEL_ORDER]
)
metrics = pd.concat(
    [metric_tables[name] for name in MODEL_ORDER],
    ignore_index=True,
)

keys = ["dialogue_idx", "turn", "unit", "real_rank", "correct"]
turn_predictions = turn_tables[MODEL_ORDER[0]].copy()
for name in MODEL_ORDER[1:]:
    turn_predictions = turn_predictions.merge(
        turn_tables[name],
        on=keys,
        how="inner",
        validate="one_to_one",
    )

assert len(turn_predictions) == 2500
assert turn_predictions["dialogue_idx"].nunique() == 515
assert (turn_predictions["real_rank"] >= 2).sum() == 1985
assert (turn_predictions["real_rank"] == 1).sum() == 515
assert parameter_frame(models["hard_floor_005"])["prior"].min() >= 0.05 - 1e-12
assert diagnostic_rows["unprotected_solution_bkt"]["priors_at_0001"] == 142

print("Parameter comparison")
print(parameter_diagnostics.set_index("model").round(4).to_string())

for protocol in [
    "all_real_turns",
    "paper_matched_R2_plus",
    "first_real_turn_R1",
]:
    print(f"\n{protocol}")
    print(
        metrics[metrics["protocol"].eq(protocol)]
        .set_index("model")[DISPLAY_METRICS]
        .reindex(MODEL_ORDER)
        .round(4)
        .to_string()
    )


Parameter comparison
                                prior_min  prior_median  prior_max  priors_at_0001  priors_at_005  median_abs_prior_shift_from_03  learning_median  guess_median  slip_median  kcs_with_any_boundary_parameter
model                                                                                                                                                                                                         
original_bkt                        0.001        0.4882     0.9990              17              0                          0.0000           0.0475        0.2938       0.2468                               81
unprotected_solution_bkt            0.001        0.0010     0.0010             142              0                          0.4872           0.7305        0.0010       0.4158                              142
inference_only_solution_update      0.001        0.4882     0.9990              17              0                          0.0000           0.0475     

In [13]:
# Exact regression checks against notebooks 03 and 04.
saved_turns = pd.read_csv(
    RESULTS / "bkt_solution_baseline_turn_predictions.csv"
)
comparison = turn_predictions.merge(
    saved_turns[
        [
            "dialogue_idx",
            "turn",
            "original_bkt_pred",
            "solution_bkt_pred",
        ]
    ],
    on=["dialogue_idx", "turn"],
    how="inner",
    validate="one_to_one",
)
assert len(comparison) == 2500
assert np.allclose(
    comparison["original_bkt"],
    comparison["original_bkt_pred"],
    atol=1e-12,
)
assert np.allclose(
    comparison["unprotected_solution_bkt"],
    comparison["solution_bkt_pred"],
    atol=1e-12,
)
print("notebook 03/04 predictions and all evaluation masks reproduced exactly")


notebook 03/04 predictions and all evaluation masks reproduced exactly


## 11. Overall interpretation

| Approach | Prevents blanket prior collapse? | Main cost |
| --- | --- | --- |
| Inference-only update | Yes; parameters are not refitted | Severe initial downward update and train–inference mismatch |
| Hard floor | Technically yes | All priors collapse to the new arbitrary floor |
| Fixed original prior | Yes, exactly | Learning, guess, and slip become extreme |
| Shrinkage | Yes; priors remain differentiated | Other parameters remain extreme; strength must be validated |

No approach restores notebook 03's all-real or paper-matched performance. The
best augmented paper-matched AUC is `0.6212` from inference-only updating,
compared with `0.6402` for the original BKT.

The evidence supports a specific conclusion:

> Prior collapse can be mechanically or statistically prevented, but it is not
> the root problem. The stronger problem is treating every later dialogue KC as
> an ordinary incorrect observation in the initial solution.

The methods in this notebook should therefore be interpreted as diagnostic
controls. If one is developed further, its hyperparameters must be selected
with dialogue-grouped validation inside training data rather than from this
test comparison.


## 12. Save auditable outputs

All fitted controlled models, per-approach diagnostics, protocol metrics, and
turn-level predictions are saved. Reload checks verify that each controlled
model reproduces its predictions exactly.


In [14]:
controlled_models = {
    name: models[name] for name in CONTROL_CONFIGS
}

model_path = MODELS / "bkt_prior_stabilization.json"
metrics_path = RESULTS / "bkt_prior_stabilization_metrics.csv"
predictions_path = RESULTS / "bkt_prior_stabilization_turn_predictions.csv"
diagnostics_path = (
    RESULTS / "bkt_prior_stabilization_parameter_diagnostics.csv"
)

payload = {
    "model_type": "prior controls for retrospective solution-augmented BKT",
    "param_names": list(bkt.PARAM_NAMES),
    "metadata": {
        "saved_utc": datetime.now(timezone.utc).isoformat(),
        "fit_config": FIT_CONFIG,
        "control_configs": CONTROL_CONFIGS,
        "fit_seconds": fit_times,
        "reference_model": "extension/models/bkt_original.json",
        "unprotected_model": "extension/models/bkt_solution_baseline.json",
        "solution_contract": (
            "correct=False; ordered union of all dialogue KCs; included in "
            "state updates; excluded from metrics"
        ),
        "selection_warning": (
            "floor and shrinkage strength are declared sensitivities, not "
            "selected from test performance"
        ),
    },
    "models": {
        name: {
            "per_skill": model.per_skill,
            "fallback": model.fallback,
        }
        for name, model in controlled_models.items()
    },
}
with model_path.open("w") as handle:
    json.dump(payload, handle, indent=2)

metrics.to_csv(metrics_path, index=False)
parameter_diagnostics.to_csv(diagnostics_path, index=False)

turn_predictions = turn_predictions.copy()
turn_predictions["in_all_real_evaluation"] = True
turn_predictions["in_paper_matched_evaluation"] = (
    turn_predictions["real_rank"] >= 2
)
turn_predictions["in_first_real_turn_evaluation"] = (
    turn_predictions["real_rank"] == 1
)
turn_predictions.to_csv(predictions_path, index=False)

with model_path.open() as handle:
    reloaded_payload = json.load(handle)
for name, saved in reloaded_payload["models"].items():
    reloaded = bkt.FittedBKT(
        saved["per_skill"],
        saved["fallback"],
    )
    reloaded_turns = aggregate_real_turns(
        reloaded.predict_long(test_long),
        "reloaded",
    )
    merged = turn_predictions.merge(
        reloaded_turns,
        on=keys,
        how="inner",
        validate="one_to_one",
    )
    assert np.allclose(merged[name], merged["reloaded"], atol=1e-12)

print("saved:", model_path)
print("saved:", metrics_path)
print("saved:", predictions_path)
print("saved:", diagnostics_path)
print("all controlled models reload exactly")


saved: extension/models/bkt_prior_stabilization.json
saved: extension/results/bkt_prior_stabilization_metrics.csv
saved: extension/results/bkt_prior_stabilization_turn_predictions.csv
saved: extension/results/bkt_prior_stabilization_parameter_diagnostics.csv
all controlled models reload exactly


## Summary

- Every approach is tested and interpreted in its own section.
- Inference-only updating protects parameters but still damages predictions.
- A hard floor relocates the prior boundary.
- Fixed priors transfer pressure into learning, guess, and slip.
- Shrinkage produces differentiated priors but does not restore overall
  performance.
- The original correctness-only BKT remains the effective baseline.
- The experiment diagnoses the solution-wide false KC union as the deeper
  modelling problem.
